####
# ECH Adaptation and Distinct ECH Configs by Test_Code for all Tests
####

Adjusted Cloudflae count    

In [ ]:
import os
import pickle
from datetime import datetime

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.pool import QueuePool

# Load environment variables from .env file
load_dotenv("../.env")  # Adjust the path if your .env file is in a different location

testDates = [
    "2024-08-08",
    "2024-08-19",
    "2024-08-21",
    "2024-08-29",
    "2024-09-05",
    "2024-09-06",
    "2024-09-08",
    "2024-09-11",
    "2024-09-15",
    "2024-10-13",
    "2024-10-20",
    "2024-10-29",
    "2024-10-31",
    "2024-11-07",
    "2024-11-12",
    "2024-11-15",
    "2024-11-17",
    "2024-11-18",
    "2024-11-19",
    "2024-11-27",
    "2024-11-28",
    "2024-11-29",
    "2024-11-30",
    "2024-12-01",
    "2024-12-02",
    "2024-12-03",
    "2024-12-04",
    "2024-12-12",
    "2024-12-13",
    "2024-12-14",
    "2024-12-15",
    "2024-12-16",
    "2024-12-17",
    "2024-12-18",
    "2024-12-19",
    "2024-12-20",
    "2024-12-21",
    "2024-12-22",
    "2024-12-23",
    "2024-12-24",
    "2024-12-25",
    "2024-12-26",
    "2024-12-29",
    "2024-12-30",
    "2025-01-01",
    "2025-01-02",
    "2025-01-03",
    "2025-01-04",
    "2025-01-05",
    "2025-01-06",
    "2025-01-07",
    "2025-01-08",
    "2025-01-09",
    "2025-01-10",
    "2025-01-11",
    "2025-01-12",
    "2025-01-13",
    "2025-01-14",
    "2025-01-15",
    "2025-01-16",
    "2025-01-17",
    "2025-01-18",
    "2025-01-19",
    "2025-01-20",
    "2025-01-21",
    "2025-01-22",
    "2025-01-23",
    "2025-01-24",
    "2025-01-25",
    "2025-01-26",
    "2025-01-27",
    "2025-01-28",
    "2025-01-29",
    "2025-01-30",
    "2025-01-31",
    "2025-02-01",
    "2025-02-02",
    "2025-02-03",
    "2025-02-04",
    "2025-02-05",
    "2025-02-06",
    "2025-02-07",
    "2025-02-08",
    "2025-02-09",
    "2025-02-10",
    "2025-02-11",
    "2025-02-12",
    "2025-02-14",
    "2025-02-15",
    "2025-02-16",
    "2025-02-17",
    "2025-02-18",
    "2025-02-19",
]

queries = {
    "https_records": """
    SELECT 
        hr.test_date,
        COUNT(CASE WHEN hr.ech_config IS NOT NULL AND hr.ech_config != '' THEN 1 END) AS total_ech_configs,
        COUNT(DISTINCT CASE WHEN hr.ech_config IS NOT NULL AND hr.ech_config != '' THEN hr.ech_config END) AS distinct_ech_configs
    FROM 
        public."HTTPSRecords" hr
    WHERE hr.test_date = %s
    GROUP BY 
        hr.test_date;
    """,
    "raw_not_empty": """
    SELECT 
        test_date, 
        COUNT(*) as raw_not_empty_count
    FROM 
        public."HTTPSRecords" 
    WHERE test_date = %s AND raw != ''
    GROUP BY test_date;
    """,
    "dns_results": """
    SELECT 
        test_date, 
        COUNT(*) as dns_results_count 
    FROM 
        public."DNSResults" 
    WHERE test_date = %s
    GROUP BY test_date;
    """,
    "cloudflare_ech": """
        SELECT 
        hr.test_date, 
        COUNT(*) AS cloudflare_https_record_count
    FROM 
        public."HTTPSRecords" hr
    JOIN 
        public."HTTPSRecordsECHConfigs" hre
    ON 
        hr.id = hre.https_record_id
    JOIN 
        public."ECHConfigs" ec
    ON 
        hre.ech_config_id = ec.id
    WHERE 
        ec.ech_public_name = 'cloudflare-ech.com' 
        AND hr.test_date = %s
    GROUP BY 
        hr.test_date;
    """,
}

# Create engine at the module level
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}@{os.getenv('DB_HOST')}/{os.getenv('DB_NAME')}",
    connect_args={"connect_timeout": 3600},
    poolclass=QueuePool,
    pool_size=5,
    max_overflow=10,
)


def fetch_data(query_name, query, date):
    """Fetches data for a single query and date."""
    print(f"Running query: {query_name} for date: {date}", flush=True)
    with engine.connect() as conn:
        params = (date,)
        df = pd.read_sql_query(query, conn, params=params)
    return df


def fetch_and_save_data(queries, filename_suffix, testDates):
    """
    Fetches data from PostgreSQL, saves results for each test date to a pickle file,
    and prints filenames for reference.

    Args:
        queries (dict): A dictionary of SQL queries to execute, with query names as keys.
        filename_suffix (str): A string to be added to the pickle filename.
        testDates (list): A list of dates to execute the queries for.
    """
    os.makedirs("./pickle", exist_ok=True)
    filenames = []

    for date in testDates:
        dfs = {}

        try:
            # Execute each query for the current test date
            for query_name, query in queries.items():
                dfs[query_name] = fetch_data(query_name, query, date)

            # Merge the dataframes on test_date
            final_df = dfs["https_records"]
            final_df = final_df.merge(dfs["raw_not_empty"], on="test_date", how="left")
            final_df = final_df.merge(dfs["dns_results"], on="test_date", how="left")
            final_df = final_df.merge(dfs["cloudflare_ech"], on="test_date", how="left")

            # Save the dataframe to a pickle file
            filename = f"./pickle/{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}_{filename_suffix}_{date}.pickle"
            filenames.append(filename)

            with open(filename, "wb") as f:
                pickle.dump(final_df, f)

            print(f"Saved pickle file: {filename}", flush=True)

        except Exception as e:
            print(f"Error processing test_date {date}: {e}", flush=True)

    # Print all filenames for easy reference
    print("\nAll generated pickle files:", flush=True)
    for filename in filenames:
        print(filename, flush=True)


if __name__ == "__main__":
    filename_suffix = "ech_adaptation_data"
    fetch_and_save_data(queries, filename_suffix, testDates)

Running query: https_records for date: 2024-08-08
